# 1. Import

In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import HuberRegressor

import warnings
warnings.filterwarnings('ignore')

In [2]:
c_train = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\clients_train.csv')
c_test = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\clients_test.csv')
r_train = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\credit_train.csv')
r_test = pd.read_csv(r'C:\Users\COMP\ML_GreenBank_Homework\data\sample\credit_test.csv')

In [3]:
clients_cols = [
    'avg_inflow_30d',
    'has_mobile_app',
    'txn_count_30d',
    'spend_food_share'
]

risk_cols  = [
    'loan_amount', 
    'monthly_payment', 
    'credit_history_months', 
    'debt_to_income_known', 
    'loan_term_months', 
    'past_due_count_12m',
    "pred_inflow",
    "pti_ratio",
    "loan_to_inflow_ratio"
]

# 2. OOF model training

In [4]:
from sklearn.model_selection import KFold

inflow = np.zeros(len(c_train))
kfold = KFold(n_splits=4, shuffle=True, random_state=67)

for tr, ts in kfold.split(c_train):
    X_tr = c_train.loc[tr, clients_cols]
    X_ts = c_train.loc[ts, clients_cols]
    y_tr = c_train.loc[tr, "inflow_next_30d"] / 1000

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_ts = scaler.transform(X_ts)

    inner_model = HuberRegressor(alpha=10)
    inner_model.fit(X_tr, y_tr)
    inflow[ts] = inner_model.predict(X_ts) * 1000

In [5]:
dt_train = pd.DataFrame({
    "client_id": c_train["client_id"], 
    "pred_inflow": inflow
})

train = pd.merge(r_train, dt_train, on="client_id", how="left")

train["pred_inflow"] = np.clip(train["pred_inflow"], 1, None)
train["pti_ratio"] = train["monthly_payment"] / train["pred_inflow"]
train["loan_to_inflow_ratio"] = train["loan_amount"] / train["pred_inflow"]

In [6]:
X_train = train[risk_cols]
y_train = train["overdue_next_60d"]

risk_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    max_depth=7,
    auto_class_weights="Balanced",
    verbose=100,
    random_state=67,
)
risk_model.fit(X_train, y_train)

0:	learn: 0.6775736	total: 147ms	remaining: 43.9s
100:	learn: 0.1959855	total: 940ms	remaining: 1.85s
200:	learn: 0.0734958	total: 1.66s	remaining: 817ms
299:	learn: 0.0367878	total: 2.35s	remaining: 0us


# 3. Predictions

In [7]:
sc = StandardScaler()
X_train = sc.fit_transform(c_train[clients_cols])
X_test = sc.transform(c_test[clients_cols])
y_train = c_train['inflow_next_30d'] / 1000

inflow_model = HuberRegressor(alpha=10)
inflow_model.fit(X_train, y_train)
pred = inflow_model.predict(X_test) * 1000

In [8]:
dt_test = pd.DataFrame({
    "client_id": c_test["client_id"], 
    "pred_inflow": pred
})

test = pd.merge(r_test, dt_test, on="client_id", how="left")

test["pred_inflow"] = np.clip(test["pred_inflow"], 1, None)
test["pti_ratio"] = test["monthly_payment"] / test["pred_inflow"]
test["loan_to_inflow_ratio"] = test["loan_amount"] / test["pred_inflow"]

In [9]:
X_test = test[risk_cols]
test["overdue_start_proba"] = risk_model.predict_proba(X_test)[:, 1]

submission = test[["application_id", "overdue_start_proba"]]
display(submission.head())

,application_id,overdue_start_proba
0,Z000000,0.336586
1,Z000001,0.195022
2,Z000002,0.330151
3,Z000003,0.036086
4,Z000004,0.050248
